In [1]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW events AS
SELECT
    "case:concept:name"     AS app_id,
    "concept:name"          AS step,
    "time:timestamp"        AS ts,
    "org:resource"          AS resource,
    "lifecycle:transition"  AS lifecycle,
    EventOrigin             AS origin,
    "case:LoanGoal"         AS loan_goal,
    "case:ApplicationType"  AS app_type,
    "case:RequestedAmount"  AS requested_amount,
    OfferedAmount           AS offered_amount,
    CreditScore             AS credit_score
FROM 'data/bpi2017.parquet'
""")

print(con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT app_id) AS total_apps
FROM events
""").df())

   total_rows  total_apps
0     1202267       31509


In [2]:
# Reusable table: one row per finished application
con.execute("""
CREATE OR REPLACE VIEW app_outcomes AS
WITH outcome AS (
    SELECT app_id, MIN(step) AS outcome
    FROM events
    WHERE step IN ('A_Pending', 'A_Cancelled', 'A_Denied')
    GROUP BY app_id
),
apps AS (
    SELECT DISTINCT app_id, loan_goal, app_type, requested_amount
    FROM events
)
SELECT a.*, o.outcome
FROM apps a
JOIN outcome o USING (app_id)
""")

print("Check, finished apps:", con.sql("SELECT COUNT(*) FROM app_outcomes").fetchone()[0])

print("\nQ1. Approval by loan goal:")
print(con.sql("""
SELECT loan_goal,
       COUNT(*) AS apps,
       ROUND(100.0 * AVG(CASE WHEN outcome = 'A_Pending'   THEN 1 ELSE 0 END), 1) AS approved_pct,
       ROUND(100.0 * AVG(CASE WHEN outcome = 'A_Cancelled' THEN 1 ELSE 0 END), 1) AS cancelled_pct
FROM app_outcomes
GROUP BY loan_goal
HAVING COUNT(*) >= 500
ORDER BY approved_pct DESC
""").df().to_string(index=False))

print("\nQ2. Approval by requested amount:")
print(con.sql("""
SELECT CASE
         WHEN requested_amount = 0     THEN '0 (not given)'
         WHEN requested_amount < 5000  THEN '1. under 5k'
         WHEN requested_amount < 15000 THEN '2. 5k-15k'
         WHEN requested_amount < 30000 THEN '3. 15k-30k'
         ELSE '4. 30k and above'
       END AS amount_band,
       COUNT(*) AS apps,
       ROUND(100.0 * AVG(CASE WHEN outcome = 'A_Pending' THEN 1 ELSE 0 END), 1) AS approved_pct
FROM app_outcomes
GROUP BY amount_band
ORDER BY amount_band
""").df().to_string(index=False))

Check, finished apps: 31411

Q1. Approval by loan goal:
             loan_goal  apps  approved_pct  cancelled_pct
   Remaining debt home   835          65.1           29.3
               Unknown  2363          63.9           27.3
      Home improvement  7646          58.7           30.3
Existing loan takeover  5574          55.1           30.7
  Extra spending limit   623          53.1           31.9
                   Car  9307          51.4           36.5
Other, see explanation  2976          50.7           35.6
         Not speficied  1058          41.5           44.6

Q2. Approval by requested amount:
     amount_band  apps  approved_pct
   0 (not given)  3077          65.3
     1. under 5k    11          18.2
       2. 5k-15k 13320          48.7
      3. 15k-30k 10356          59.1
4. 30k and above  4647          56.2


In [3]:
print("Q3. Zero amount vs application type:")
print(con.sql("""
SELECT app_type,
       (requested_amount = 0) AS zero_amount,
       COUNT(*) AS apps,
       ROUND(100.0 * AVG(CASE WHEN outcome = 'A_Pending' THEN 1 ELSE 0 END), 1) AS approved_pct
FROM app_outcomes
GROUP BY app_type, zero_amount
ORDER BY app_type, zero_amount
""").df().to_string(index=False))

print("\nQ4. Number of offers vs approval:")
print(con.sql("""
WITH offers AS (
    SELECT app_id, COUNT(*) AS n
    FROM events
    WHERE step = 'O_Create Offer'
    GROUP BY app_id
)
SELECT LEAST(COALESCE(n, 0), 4) AS offers_4plus,
       COUNT(*) AS apps,
       ROUND(100.0 * AVG(CASE WHEN outcome = 'A_Pending' THEN 1 ELSE 0 END), 1) AS approved_pct
FROM app_outcomes
LEFT JOIN offers USING (app_id)
GROUP BY offers_4plus
ORDER BY offers_4plus
""").df().to_string(index=False))

print("\nQ5. Top 10 staff by completed tasks (system user excluded):")
print(con.sql("""
WITH tasks AS (
    SELECT resource, COUNT(*) AS done
    FROM events
    WHERE origin = 'Workflow'
      AND lifecycle = 'complete'
      AND resource <> 'User_1'
    GROUP BY resource
)
SELECT resource,
       done,
       ROUND(100.0 * done / SUM(done) OVER (), 1) AS share_pct,
       RANK() OVER (ORDER BY done DESC) AS rank
FROM tasks
ORDER BY done DESC
LIMIT 10
""").df().to_string(index=False))

print("\nStaff with at least one completed task:",
      con.sql("""
      SELECT COUNT(DISTINCT resource) FROM events
      WHERE origin = 'Workflow' AND lifecycle = 'complete' AND resource <> 'User_1'
      """).fetchone()[0])

Q3. Zero amount vs application type:
   app_type  zero_amount  apps  approved_pct
Limit raise        False  2621          72.9
Limit raise         True   766          74.9
 New credit        False 25713          51.8
 New credit         True  2311          62.1

Q4. Number of offers vs approval:
 offers_4plus  apps  approved_pct
            1 22900          53.2
            2  6549          57.6
            3  1337          66.1
            4   625          62.6

Q5. Top 10 staff by completed tasks (system user excluded):
resource  done  share_pct  rank
 User_68  1797        4.3     1
 User_30  1660        4.0     2
 User_99  1644        3.9     3
 User_29  1353        3.2     4
 User_87  1216        2.9     5
 User_83  1117        2.7     6
 User_95  1110        2.7     7
  User_3  1097        2.6     8
 User_49  1010        2.4     9
 User_75   839        2.0    10

Staff with at least one completed task: 142


## What SQL showed

I moved the data into DuckDB and gave the columns shorter names so the queries are easier to read. Before trusting anything, I checked the totals matched pandas: same 1,202,267 rows and 31,509 applications.

Loan purpose matters. Applications to pay off remaining debt on a home were approved 65% of the time. The ones where the customer didn't specify a purpose were approved only 42% of the time, and almost half of them were cancelled.

Larger requests were approved more often. Between 15k and 30k, 59% were approved, compared with 49% for 5k to 15k. I'd be careful here though. It could simply be that people asking for more are stronger applicants to begin with.

Existing customers asking for a limit raise were approved about 73% of the time, against 52% for new credit.

I couldn't explain the applications with a requested amount of 0. I first thought they were incomplete forms, but they actually get approved more than average. Then I thought they might be limit raises, but three quarters of them are new credit. So I'm leaving this as an open question for the bank rather than guessing.

Every finished application received at least one offer. Customers who got three offers were approved 66% of the time, compared with 53% for just one.

Work isn't spread evenly. Out of 142 staff, the top 10 completed about 31% of all tasks. The busiest person handled roughly six times the average. That might just reflect different roles or part-time staff, which the data doesn't show, but it's worth asking about.